# Notebook 02 — Benchmarking e Treinamento de Modelos de Visão

## Objetivos

1. Implementar cálculos teóricos de FLOPs para convoluções e atenção
2. Construir e comparar CNN simples vs CNN com depthwise separable convolutions
3. Realizar benchmark de latência ao vivo entre diferentes arquiteturas
4. Treinar modelos em dataset sintético de imagens e comparar performance
5. Analisar impacto da resolução na latência de forma empírica

## Fundamentação Teórica

Conforme seção 2 do material (*Arquiteturas CNN Eficientes*):

> MobileNet introduziu **Depthwise Separable Convolutions** como alternativa às
> convoluções padrão. Esta técnica decompõe uma convolução tradicional em duas
> operações: uma convolução depthwise (filtro por canal) seguida de pointwise (1×1).
> A redução de FLOPs é:
>
> $$\text{Redução} = \frac{1}{C_{out}} + \frac{1}{K^2}$$
>
> Para K=3, redução ≈ 8-9x. — Howard et al. (2017)

Neste notebook, implementamos e verificamos essas propriedades na prática.

In [ ]:
# Imports
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import (
    SimpleCNN,
    DepthwiseSeparableCNN,
    VisionBenchmarkModel,
    calculate_conv_flops,
    calculate_attention_flops,
    calculate_depthwise_separable_reduction,
    calculate_n_patches,
    calculate_receptive_field,
)
from src.data_preprocessing import DataPreprocessor
from src.training import train_model, create_dataloaders, benchmark_architectures
from src.evaluation import plot_training_history, plot_benchmark_comparison
from src.utils import BenchmarkTimer, set_seed, get_device, ensure_dirs

# Setup
set_seed(42)
ensure_dirs('..')
device = get_device()
print(f'Dispositivo: {device}')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## 1. Cálculos Teóricos de FLOPs

Implementação direta das fórmulas da seção 1 do material:

- **Convolução padrão**: FLOPs = 2 × K² × C_in × C_out × H_out × W_out
- **Self-attention**: FLOPs = 4 × n × d² + 2 × n² × d

In [ ]:
# Exemplo: FLOPs de uma camada conv 3×3 (64→128) com feature map 56×56
flops_conv = calculate_conv_flops(kernel_size=3, c_in=64, c_out=128, h_out=56, w_out=56)
print(f'FLOPs conv 3×3 (64→128, 56×56): {flops_conv:,} ({flops_conv/1e6:.1f} MFLOPs)')

# Comparação: conv 7×7 vs conv 3×3 (ambas 64→128, 56×56)
flops_7x7 = calculate_conv_flops(kernel_size=7, c_in=64, c_out=128, h_out=56, w_out=56)
print(f'FLOPs conv 7×7 (64→128, 56×56): {flops_7x7:,} ({flops_7x7/1e6:.1f} MFLOPs)')
print(f'Razão 7×7 / 3×3: {flops_7x7 / flops_conv:.1f}x (teórico: {7**2/3**2:.1f}x)')

# FLOPs de self-attention para ViT
n_patches_224 = calculate_n_patches(224, patch_size=16)  # 196 patches
n_patches_448 = calculate_n_patches(448, patch_size=16)  # 784 patches

flops_att_224 = calculate_attention_flops(n_patches_224, d_model=384)
flops_att_448 = calculate_attention_flops(n_patches_448, d_model=384)

print(f'\nSelf-attention (DeiT-Small, d=384):')
print(f'  224×224 ({n_patches_224} patches): {flops_att_224:,} ({flops_att_224/1e6:.1f} MFLOPs)')
print(f'  448×448 ({n_patches_448} patches): {flops_att_448:,} ({flops_att_448/1e6:.1f} MFLOPs)')
print(f'  Razão 448/224: {flops_att_448/flops_att_224:.1f}x (patches 4x → atenção ≈16x)')

## 2. Redução com Depthwise Separable Convolutions

Conforme Howard et al. (2017), a redução de FLOPs é:

$$\text{Redução} = \frac{1}{C_{out}} + \frac{1}{K^2}$$

In [ ]:
# Calcular redução para diferentes configurações
print('=== Redução FLOPs com Depthwise Separable Conv ===')
print(f'{"K":>4} | {"C_out":>6} | {"Redução":>8} | {"Speedup":>8}')
print('-' * 40)

for k in [3, 5, 7]:
    for c_out in [32, 64, 128, 256]:
        reduction = calculate_depthwise_separable_reduction(c_out, k)
        speedup = 1.0 / reduction
        print(f'{k:>4} | {c_out:>6} | {reduction:>8.4f} | {speedup:>8.1f}x')

# Verificação empírica: contar parâmetros de conv padrão vs depthwise separable
conv_standard = nn.Conv2d(64, 128, kernel_size=3, padding=1)
conv_dw = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64)  # depthwise
conv_pw = nn.Conv2d(64, 128, kernel_size=1)  # pointwise

params_std = sum(p.numel() for p in conv_standard.parameters())
params_dw_pw = sum(p.numel() for p in conv_dw.parameters()) + sum(p.numel() for p in conv_pw.parameters())

print(f'\n=== Verificação de Parâmetros (64→128, K=3) ===')
print(f'Conv padrão:       {params_std:>8,} parâmetros')
print(f'Depthwise+Pointwise: {params_dw_pw:>8,} parâmetros')
print(f'Redução real:      {params_dw_pw/params_std:.3f} ({1/(params_dw_pw/params_std):.1f}x speedup)')

## 3. Construir Modelos: CNN Simples vs MobileNet-Style

Criamos dois modelos pedagógicos para comparação direta:
- **SimpleCNN**: convoluções 3×3 padrão
- **DepthwiseSeparableCNN**: estilo MobileNet com depthwise separable convolutions

In [ ]:
# Instanciar modelos
simple_cnn = SimpleCNN(n_classes=4)
mobile_cnn = DepthwiseSeparableCNN(n_classes=4)

# Contar parâmetros
params_simple = sum(p.numel() for p in simple_cnn.parameters())
params_mobile = sum(p.numel() for p in mobile_cnn.parameters())

print(f'SimpleCNN:            {params_simple:>10,} parâmetros ({params_simple/1e6:.2f}M)')
print(f'DepthwiseSeparableCNN: {params_mobile:>10,} parâmetros ({params_mobile/1e6:.2f}M)')
print(f'Razão: {params_mobile/params_simple:.3f}x')

# Estimar FLOPs com nosso wrapper
bm_simple = VisionBenchmarkModel('simple_cnn', pretrained=False, device=device)
bm_mobile = VisionBenchmarkModel('mobilenet_style', pretrained=False, device=device)

flops_simple = bm_simple.estimate_flops(input_size=64)
flops_mobile = bm_mobile.estimate_flops(input_size=64)

print(f'\nFLOPs @ 64×64:')
print(f'  SimpleCNN:             {flops_simple["total_mflops"]:>8.2f} MFLOPs')
print(f'  DepthwiseSeparableCNN: {flops_mobile["total_mflops"]:>8.2f} MFLOPs')
if flops_simple["total_flops"] > 0:
    print(f'  Razão: {flops_mobile["total_flops"]/flops_simple["total_flops"]:.3f}x')

## 4. Benchmark de Latência: CNN Simples vs MobileNet-Style

Conforme snippet 1 do Hands On do material, realizamos benchmark
comparativo de latência com warm-up e múltiplas execuções.

In [ ]:
# Benchmark de latência em diferentes resoluções
resolutions = [32, 64, 128, 224]
results_simple = []
results_mobile = []

for res in resolutions:
    # SimpleCNN
    stats_s = bm_simple.benchmark_latency(input_size=res, n_warmup=5, n_runs=50)
    results_simple.append({'resolution': res, **stats_s})
    
    # DepthwiseSeparableCNN
    stats_m = bm_mobile.benchmark_latency(input_size=res, n_warmup=5, n_runs=50)
    results_mobile.append({'resolution': res, **stats_m})
    
    print(f'Res {res:>3}: SimpleCNN={stats_s["mean_ms"]:.2f}ms | MobileStyle={stats_m["mean_ms"]:.2f}ms')

# Plotar comparação
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(resolutions, [r['mean_ms'] for r in results_simple], 'o-', label='SimpleCNN', color='#2196F3')
ax1.plot(resolutions, [r['mean_ms'] for r in results_mobile], 's-', label='MobileNet-Style', color='#F44336')
ax1.set_xlabel('Resolução de Entrada')
ax1.set_ylabel('Latência (ms)')
ax1.set_title('Latência vs Resolução')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(resolutions, [r['throughput_imgs_sec'] for r in results_simple], 'o-', label='SimpleCNN', color='#2196F3')
ax2.plot(resolutions, [r['throughput_imgs_sec'] for r in results_mobile], 's-', label='MobileNet-Style', color='#F44336')
ax2.set_xlabel('Resolução de Entrada')
ax2.set_ylabel('Throughput (imgs/s)')
ax2.set_title('Throughput vs Resolução')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/benchmark_simple_vs_mobile.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Campo Receptivo

Conforme seção 9 do material:

> $$RF_l = RF_{l-1} + (k_l - 1) \times \prod_{i=1}^{l-1} s_i$$
>
> Strides > 1 amplificam dramaticamente o crescimento do campo receptivo.

In [ ]:
# Comparar campo receptivo de diferentes configurações
configs = {
    '3 camadas conv3×3, stride=1': {
        'kernels': [3, 3, 3],
        'strides': [1, 1, 1],
    },
    '3 camadas conv3×3, stride=2': {
        'kernels': [3, 3, 3],
        'strides': [2, 2, 2],
    },
    'VGG-style (5 camadas)': {
        'kernels': [3, 3, 3, 3, 3],
        'strides': [1, 2, 1, 2, 1],
    },
    'ResNet-style (conv7 + 4×conv3)': {
        'kernels': [7, 3, 3, 3, 3],
        'strides': [2, 2, 1, 2, 1],
    },
}

print('=== Campo Receptivo Efetivo ===')
for name, cfg in configs.items():
    rf = calculate_receptive_field(cfg['kernels'], cfg['strides'])
    print(f'{name:>40}: RF = {rf} pixels')

# Comparar com ViT (campo receptivo global desde camada 1)
print(f'\nViT (qualquer camada de atenção):         RF = GLOBAL ({224}×{224} = toda imagem)')

## 6. Gerar Dataset Sintético de Imagens

Geramos imagens sintéticas simulando inspeção industrial (caso Siemens mencionado
na seção 'Mercado, Cases e Tendências') para treinar nossos modelos pedagógicos.

In [ ]:
# Gerar imagens sintéticas
images, labels = DataPreprocessor.generate_synthetic_images(
    output_dir='../data/raw/images',
    n_samples=2000,
    img_size=64,
    n_classes=4,
    random_state=42,
)

print(f'Imagens: {images.shape} (N, C, H, W)')
print(f'Labels: {labels.shape}, classes: {np.unique(labels)}')
print(f'Distribuição: {dict(zip(*np.unique(labels, return_counts=True)))}')

# Visualizar amostras
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
class_names = ['Normal (Círculo)', 'Defeito (Arranhão)', 'Tipo A (Triângulo)', 'Anomalia']
for cls in range(4):
    idx = np.where(labels == cls)[0][:2]
    for j, i in enumerate(idx):
        ax = axes[j, cls]
        # Transpor de (C, H, W) para (H, W, C) para visualização
        img = np.transpose(images[i], (1, 2, 0))
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(f'Classe {cls}: {class_names[cls]}', fontsize=9)
        ax.axis('off')

plt.suptitle('Amostras do Dataset Sintético de Inspeção Industrial', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Treinamento: SimpleCNN vs DepthwiseSeparableCNN

Treinamos ambos os modelos no mesmo dataset e comparamos:
- Velocidade de convergência (tempo por época)
- Acurácia final
- Trade-off latência vs acurácia

In [ ]:
# Criar DataLoaders
train_loader, val_loader = create_dataloaders(
    images, labels, batch_size=32, val_split=0.2, random_state=42,
)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

# Treinar SimpleCNN
print('\n=== Treinamento: SimpleCNN ===')
model_simple = SimpleCNN(n_classes=4)
history_simple = train_model(
    model_simple, train_loader, val_loader,
    n_epochs=10, lr=1e-3, device=device,
)

In [ ]:
# Treinar DepthwiseSeparableCNN
print('\n=== Treinamento: DepthwiseSeparableCNN ===')
model_mobile = DepthwiseSeparableCNN(n_classes=4)
history_mobile = train_model(
    model_mobile, train_loader, val_loader,
    n_epochs=10, lr=1e-3, device=device,
)

In [ ]:
# Comparar curvas de treinamento
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, 11)

# Loss
axes[0].plot(epochs, history_simple['val_loss'], 'o-', label='SimpleCNN', color='#2196F3')
axes[0].plot(epochs, history_mobile['val_loss'], 's-', label='MobileNet-Style', color='#F44336')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Loss de Validação')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history_simple['val_acc'], 'o-', label='SimpleCNN', color='#2196F3')
axes[1].plot(epochs, history_mobile['val_acc'], 's-', label='MobileNet-Style', color='#F44336')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Acurácia de Validação')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Tempo por época
axes[2].bar(['SimpleCNN', 'MobileNet-Style'],
            [np.mean(history_simple['epoch_times']),
             np.mean(history_mobile['epoch_times'])],
            color=['#2196F3', '#F44336'])
axes[2].set_ylabel('Tempo Médio por Época (s)')
axes[2].set_title('Eficiência de Treinamento')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Comparação: CNN Padrão vs Depthwise Separable CNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumo
print('\n=== Resumo do Treinamento ===')
print(f'SimpleCNN:    Val Acc = {history_simple["val_acc"][-1]:.4f} | Tempo total = {history_simple["total_time"]:.2f}s')
print(f'MobileStyle:  Val Acc = {history_mobile["val_acc"][-1]:.4f} | Tempo total = {history_mobile["total_time"]:.2f}s')

## 8. Salvar Modelos Treinados

Salvamos os checkpoints para uso no notebook de avaliação.

In [ ]:
from src.utils import save_model

# Salvar modelos
save_model(
    model_simple,
    '../outputs/models/simple_cnn.pt',
    metadata={'val_acc': history_simple['val_acc'][-1], 'epochs': 10},
)
save_model(
    model_mobile,
    '../outputs/models/mobilenet_style.pt',
    metadata={'val_acc': history_mobile['val_acc'][-1], 'epochs': 10},
)
print('Modelos salvos em outputs/models/')

## 9. Benchmark com Modelos timm (opcional)

Conforme snippet 1 do Hands On do material:

```python
efficientnet = timm.create_model('efficientnet_b0', pretrained=True).eval().cuda()
deit = timm.create_model('deit_small_patch16_224', pretrained=True).eval().cuda()
```

Se timm estiver disponível, comparamos modelos pré-treinados.

In [ ]:
# Benchmark com timm (se disponível)
try:
    import timm
    print('timm disponível — executando benchmark com modelos pré-treinados\n')
    
    timm_models = ['efficientnet_b0', 'mobilenetv2_100']
    timm_results = benchmark_architectures(
        model_names=timm_models,
        input_sizes=[128, 224],
        n_runs=20,
        device=device,
    )
    
    # Adicionar modelos pedagógicos
    custom_results = benchmark_architectures(
        model_names=['simple_cnn', 'mobilenet_style'],
        input_sizes=[128, 224],
        n_runs=20,
        device=device,
    )
    
    all_results = timm_results + custom_results
    fig = plot_benchmark_comparison(all_results, save_path='../outputs/figures/benchmark_all.png')
    plt.show()
    
except ImportError:
    print('timm não instalado — usando apenas modelos pedagógicos')
    all_results = benchmark_architectures(
        model_names=['simple_cnn', 'mobilenet_style'],
        input_sizes=[64, 128, 224],
        n_runs=20,
        device=device,
    )
    fig = plot_benchmark_comparison(all_results, save_path='../outputs/figures/benchmark_custom.png')
    plt.show()

## Resumo

Neste notebook:

1. **Calculamos FLOPs** usando as fórmulas teóricas da seção 1 do material
2. **Verificamos empiricamente** a redução de ~8x com depthwise separable convolutions (Howard et al., 2017)
3. **Comparamos campo receptivo** de diferentes configurações arquiteturais
4. **Treinamos modelos** SimpleCNN vs MobileNet-Style em dataset de inspeção industrial
5. **Benchmarkamos latência** em diferentes resoluções, confirmando crescimento com resolução
6. **Salvamos modelos** para avaliação no próximo notebook

**Próximo notebook:** [03_avaliacao.ipynb](03_avaliacao.ipynb) — Fronteira de Pareto,
keyframe extraction e processamento de vídeo otimizado.